# Ratings.CSV 

Este notebook documenta el análisis exploratorio y limpieza de **ratings.csv** de MovieLens.


**Entrada**: ratings.csv \
**Objetivos**: lectura, validación, limpieza y transformación \
**Salida**: ratings_clean.parquet


## Descripción del proceso

**Análisis y comprensión**

El dataset contiene cuatro columnas: `userId`, identificador único del usuario; `movieId`, identificador entero de la película; `rating`, la valoración en formato decimal; y `timestamp`, la fecha de la valoración expresada en segundos, con valores comprendidos entre "1996-03-26" y "2018-09-26".

**Validación**

Se comprueba que la combinación de `userId` y `movieId` sea única, que las fechas estén dentro del rango entre "1996-03-26" y "2018-09-26", y que `rating` esté incluido en el intervalo [0.5, 5] con saltos de 0.5.

**Limpieza**

Si existieran registros duplicados de un mismo usuario y película, se conservaría el más reciente. No se permiten valores nulos, por lo que el registro afectado se eliminaría. Los ratings fuera del rango válido también se eliminarían.

**Transformación**

Se convierte `timestamp` a formato datetime y se guarda el resultado en el archivo ratings.parquet.


In [12]:
import pandas as pd
import numpy as np

## Análisis y comprensión del dataset

Se cargan los datos de `ratings.csv` y se revisan los tipos de cada columna y la presencia de valores nulos.

In [13]:
# Lectura de datos
ratings = pd.read_csv("../data/01_raw/movielens/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [14]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Los tipos de datos son correctos y no hay valores nulos en ninguna columna. `timestamp` se mantiene como entero por ahora; su conversión a fecha se realiza en la transformación.

## Validación

Se comprueba que no existan registros duplicados, que las fechas estén dentro del rango esperado y que los ratings respeten la escala válida.

Registros duplicados: la combinación de `userId` y `movieId` debe identificar de forma única cada valoración.

In [15]:
ratings[ratings[['userId', 'movieId']].duplicated()]
# no se observan

,userId,movieId,rating,timestamp


El resultado está vacío, por lo que no se han encontrado registros duplicados en la combinación `userId` y `movieId`.

Validación de fechas. La columna `timestamp` viene en formato Unix, es decir, en segundos transcurridos desde el 1 de enero de 1970. Se convierte a fecha para comprobar que todos los registros están dentro del rango esperado.

In [16]:
fechas = pd.to_datetime(ratings['timestamp'], unit="s")  
fechas.between("1996-03-26", "2018-09-26", inclusive="both")
fechas[~fechas.between("1996-03-26", "2018-09-26", inclusive="both")]
#todas los registros de fechas dentro del en rango contemplado

Series([], Name: timestamp, dtype: datetime64[s])

La serie resultante está vacía, lo que confirma que todos los registros tienen fechas dentro del rango esperado.

Validación de la escala de ratings. Los valores deben estar dentro del intervalo [0.5, 5] con saltos de 0.5.

In [17]:
escalaRating = np.arange(0.5,5.5,0.5)
ratings[~ratings['rating'].isin(escalaRating)]
# todos los registros cumplen la regla

,userId,movieId,rating,timestamp


El resultado está vacío, por lo que todos los ratings cumplen la escala válida.

## Limpieza

La validación anterior no ha encontrado duplicados, fechas fuera de rango ni ratings inválidos. Aun así, a continuación se muestran las comprobaciones y operaciones de limpieza que se aplicarían si existieran esos casos.

Si hubiese duplicados, se conservaría el registro más reciente ordenando por `timestamp`. Como ya se ha comprobado que no existen duplicados, esta operación no elimina ningún registro en este dataset.

In [18]:
ratings.sort_values('timestamp').drop_duplicates(subset=['movieId', 'userId'], keep='last')

,userId,movieId,rating,timestamp
66669,429,165,4.0,828124615
66719,429,595,5.0,828124615
66713,429,434,4.0,828124615
66717,429,590,5.0,828124615
66716,429,588,5.0,828124615
...,...,...,...,...
81475,514,187031,2.5,1537674927
81477,514,187595,3.0,1537674946
81336,514,5247,2.5,1537757040
81335,514,5246,1.5,1537757059


Este resultado solo se muestra a modo de comprobación y no se reasigna a `ratings`, ya que no hay duplicados que eliminar.

No se permiten valores nulos ni NA; el registro afectado se eliminaría. Se comprueba con `isnull()` e `isna()`, dos métodos equivalentes en pandas, para confirmar que no hay valores faltantes.

In [19]:
ratings.isnull().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [20]:
ratings.isna().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [21]:
ratings.dropna()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


Al igual que en el paso anterior, este resultado solo se muestra a modo de comprobación y no se reasigna a `ratings`, ya que no hay valores nulos que eliminar.

## Transformación

Se convierte `timestamp` de segundos Unix a formato datetime. Como la validación previa ha confirmado que no hay duplicados, fechas fuera de rango ni valores nulos, el resultado se guarda directamente en `ratings_clean.parquet`.

In [22]:
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit="s")
ratings.to_parquet('../data/02_processed/ratings_clean.parquet', index=False)